# 16 validate cnv consensus

**Purpose:** Validate CNV-supported compartments and cluster calls.

**Original analysis notebook:** `16_final_visual_QC_and_statistical_validation_CNV_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 35 — Final visual QC and statistical validation for marker + CopyKAT CNV consensus

This notebook performs the next thesis-ready validation step after the final CopyKAT/marker/infercnvpy consensus.

It focuses on:

1. Final UMAP QC plots for malignancy group, CopyKAT call, cluster, annotation, dataset and condition.
2. Cluster-level and group-level CopyKAT enrichment statistics.
3. Tumor-like/cycling clusters versus core reference clusters Fisher tests.
4. Sample-level condition comparisons to avoid relying only on per-cell p-values.
5. Exported TSVs and PNGs for review.

The notebook reads the final consensus h5ad in backed mode, so it should not load the full expression matrix into RAM.

In [ ]:
# =========================
# CONFIG
# =========================
from pathlib import Path
import os
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scipy.stats import fisher_exact, chi2_contingency, kruskal, mannwhitneyu

try:
    import anndata as ad
except Exception as e:
    raise ImportError("Please install/load anndata in this Python environment.") from e

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
FIG_DIR = PROJECT_DIR / "figures" / "final_CNV_QC_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

RUN_TAG = "HVG8000_r0_6"

FINAL_H5AD_CANDIDATES = [
    PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad",
    PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_copykat_consensus.h5ad",
    PROCESSED_DIR / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad",
]

CLUSTER_CONSENSUS_TSV = METADATA_DIR / f"final_copykat_marker_infercnvpy_cluster_consensus_{RUN_TAG}.tsv"
CLUSTER_COUNTS_TSV = METADATA_DIR / f"final_copykat_marker_cluster_counts_{RUN_TAG}.tsv"

# To keep plotting fast and memory-safe.
# Set to None if you want all cells plotted.
PLOT_MAX_POINTS = 250_000
RANDOM_SEED = 17

# Main clusters based on our final decision.
MALIGNANT_CLUSTERS = {"0", "4", "7", "11"}
CYCLING_CANDIDATE_CLUSTERS = {"2"}
CORE_REFERENCE_CLUSTERS = {"8", "14"}

print("Project:", PROJECT_DIR)
print("Figures:", FIG_DIR)


In [ ]:
# =========================
# HELPERS
# =========================
def first_existing(paths, label="file"):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        raise FileNotFoundError("No %s found. Checked:\n" % label + "\n".join(str(p) for p in paths))
    print(f"[USING {label}]", existing[0])
    return existing[0]


def choose_col(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            print(f"[USING {label}] {c}")
            return c
    if required:
        raise KeyError(f"None of these {label} candidates found: {candidates}\nAvailable columns: {list(df.columns)[:120]}")
    print(f"[MISSING optional {label}] candidates: {candidates}")
    return None


def write_tsv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")
    return path


def bh_fdr(pvals):
    p = np.asarray([np.nan if x is None else x for x in pvals], dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    restored = np.empty_like(q)
    restored[order] = q
    out[valid] = restored
    return out


def as_cluster_str(s):
    x = pd.Series(s).astype(str)
    x = x.str.replace(r"\.0$", "", regex=True)
    return x


def clean_copykat_pred(s):
    x = pd.Series(s).astype(str).str.strip().str.lower()
    x = x.replace({
        "not.defined": "not_defined",
        "not defined": "not_defined",
        "not_defined": "not_defined",
        "nan": "missing",
        "none": "missing",
        "": "missing",
    })
    x = x.where(x.isin(["aneuploid", "diploid", "not_defined", "missing"]), x)
    return x


def safe_odds_ratio(a, b, c, d):
    # Haldane-Anscombe only for display if zero cells exist.
    if min(a, b, c, d) == 0:
        return ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))
    return (a * d) / (b * c)


def cramer_v_from_table(tbl):
    chi2, p, dof, expected = chi2_contingency(tbl)
    n = np.asarray(tbl).sum()
    r, k = np.asarray(tbl).shape
    if n == 0 or min(r - 1, k - 1) <= 0:
        return np.nan, chi2, p, dof
    phi2 = chi2 / n
    return math.sqrt(phi2 / min(k - 1, r - 1)), chi2, p, dof


## 1. Load final consensus h5ad and merge cluster-level consensus if needed

In [ ]:
FINAL_H5AD = first_existing(FINAL_H5AD_CANDIDATES, label="final h5ad")

adata = ad.read_h5ad(FINAL_H5AD, backed="r")
obs = adata.obs.copy()
obs.index = obs.index.astype(str)
print(adata)
print("obs shape:", obs.shape)
print("obsm keys:", list(adata.obsm.keys()))

# Detect main columns.
cluster_col = choose_col(obs, [
    "leiden_scvi_main",
    "leiden_scvi_HVG8000_r0_6",
    "leiden_scvi_0.6",
    "leiden_scvi_0.7",
    "cluster",
], label="cluster")

annotation_col = choose_col(obs, [
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "cell_type_marker_based",
    "annotation",
], required=False, label="annotation")

copykat_col = choose_col(obs, [
    "copykat_pred_clean",
    "copykat_microbatch_pred_clean",
    "copykat_prediction_clean",
    "copykat.pred",
    "copykat_pred",
    "copykat_call",
], required=False, label="CopyKAT prediction")

final_group_col = choose_col(obs, [
    "final_malignancy_group_copykat_marker",
    "final_malignancy_group",
    "malignancy_group_copykat_marker",
    "marker_copykat_malignancy_group",
], required=False, label="final malignancy group")

condition_col = choose_col(obs, ["condition", "disease", "diagnosis", "tumor_status"], required=False, label="condition")
dataset_col = choose_col(obs, ["core_dataset", "dataset_id", "dataset", "Dataset"], required=False, label="dataset")
patient_col = choose_col(obs, ["patient_id", "patient", "Patient"], required=False, label="patient")
sample_col = choose_col(obs, ["sample_id", "sample", "Sample", "orig.ident"], required=False, label="sample")

# Standardized analysis columns.
obs["cluster_str"] = as_cluster_str(obs[cluster_col])

# Merge final cluster consensus if final group or CopyKAT column is missing/incomplete.
if CLUSTER_COUNTS_TSV.exists():
    cluster_counts = pd.read_csv(CLUSTER_COUNTS_TSV, sep="\t")
    cluster_counts["cluster_str"] = as_cluster_str(cluster_counts["cluster"])
    merge_cols = ["cluster_str"]
    for c in ["final_malignancy_group_copykat_marker", "final_cnv_consensus_by_cluster", "aneuploid_fraction_defined", "aneuploid_fraction_all"]:
        if c in cluster_counts.columns and c not in obs.columns:
            merge_cols.append(c)
    if len(merge_cols) > 1:
        obs = obs.merge(cluster_counts[merge_cols].drop_duplicates("cluster_str"), on="cluster_str", how="left", suffixes=("", "_from_cluster"))
        print("[MERGED cluster_counts]", merge_cols)

# Re-detect after merge.
if final_group_col is None:
    final_group_col = choose_col(obs, ["final_malignancy_group_copykat_marker"], label="final malignancy group after merge")
if copykat_col is None:
    copykat_col = choose_col(obs, ["copykat_pred_clean", "copykat_microbatch_pred_clean", "copykat_prediction_clean"], required=False, label="CopyKAT prediction after merge")

# If CopyKAT pred column is still missing but counts only exist, stop because cell-level stats need predictions.
if copykat_col is None:
    raise KeyError("No cell-level CopyKAT prediction column found in final h5ad. Please check final consensus h5ad obs columns.")

obs["copykat_pred_clean_for_stats"] = clean_copykat_pred(obs[copykat_col])
obs["final_group_for_stats"] = obs[final_group_col].astype(str).fillna("missing")

# Optional UMAP coords.
if "X_umap" in adata.obsm.keys():
    umap = pd.DataFrame(np.asarray(adata.obsm["X_umap"]), index=obs.index, columns=["UMAP1", "UMAP2"])
    print("UMAP shape:", umap.shape)
else:
    umap = None
    print("[WARNING] X_umap not found; UMAP plots will be skipped.")

column_report = pd.DataFrame([
    {"field": "cluster", "column": cluster_col},
    {"field": "annotation", "column": annotation_col},
    {"field": "copykat_prediction", "column": copykat_col},
    {"field": "final_malignancy_group", "column": final_group_col},
    {"field": "condition", "column": condition_col},
    {"field": "dataset", "column": dataset_col},
    {"field": "patient", "column": patient_col},
    {"field": "sample", "column": sample_col},
])
write_tsv(column_report, METADATA_DIR / f"final_CNV_QC_obs_column_report_{RUN_TAG}.tsv")

readiness = pd.DataFrame([{
    "final_h5ad": str(FINAL_H5AD),
    "n_cells": obs.shape[0],
    "n_obs_columns": obs.shape[1],
    "has_umap": umap is not None,
    "cluster_col": cluster_col,
    "copykat_col": copykat_col,
    "final_group_col": final_group_col,
}])
write_tsv(readiness, METADATA_DIR / f"final_CNV_QC_readiness_{RUN_TAG}.tsv")


## 2. Final counts and composition summaries

In [ ]:
# Overall final group and CopyKAT composition.
group_counts = (
    obs.groupby(["final_group_for_stats", "copykat_pred_clean_for_stats"], observed=True)
       .size().reset_index(name="n_cells")
       .sort_values(["final_group_for_stats", "n_cells"], ascending=[True, False])
)
group_counts["fraction_all_cells"] = group_counts["n_cells"] / len(obs)
write_tsv(group_counts, METADATA_DIR / f"final_CNV_QC_group_counts_{RUN_TAG}.tsv")

def summarize_cluster_group(df):
    g = (df.groupby(["cluster_str", "final_group_for_stats", "copykat_pred_clean_for_stats"], observed=True)
           .size().reset_index(name="n_cells"))
    # Wide for useful metrics.
    wide = g.pivot_table(index=["cluster_str", "final_group_for_stats"], columns="copykat_pred_clean_for_stats", values="n_cells", fill_value=0, aggfunc="sum").reset_index()
    for c in ["aneuploid", "diploid", "not_defined", "missing"]:
        if c not in wide.columns:
            wide[c] = 0
    wide["n_defined"] = wide["aneuploid"] + wide["diploid"]
    wide["n_total"] = wide[["aneuploid", "diploid", "not_defined", "missing"]].sum(axis=1)
    wide["aneuploid_fraction_defined"] = np.where(wide["n_defined"] > 0, wide["aneuploid"] / wide["n_defined"], np.nan)
    wide["aneuploid_fraction_all"] = np.where(wide["n_total"] > 0, wide["aneuploid"] / wide["n_total"], np.nan)
    return wide.sort_values("cluster_str", key=lambda s: pd.to_numeric(s, errors="coerce"))

cluster_group_summary = summarize_cluster_group(obs)
write_tsv(cluster_group_summary, METADATA_DIR / f"final_CNV_QC_cluster_group_copykat_summary_{RUN_TAG}.tsv")

display(group_counts)
display(cluster_group_summary.head(25))


## 3. Final UMAP visual QC

In [ ]:
figure_records = []


def plot_umap_categorical(obs_df, umap_df, color_col, title, out_png, max_points=PLOT_MAX_POINTS, seed=RANDOM_SEED):
    if umap_df is None:
        return None
    df = umap_df.join(obs_df[[color_col]].astype(str), how="inner")
    df = df.replace({color_col: {"nan": "missing", "None": "missing"}})
    if max_points is not None and len(df) > max_points:
        df = df.sample(max_points, random_state=seed)
    cats = pd.Categorical(df[color_col]).categories.tolist()
    cmap = plt.get_cmap("tab20") if len(cats) <= 20 else plt.get_cmap("hsv")
    color_map = {cat: cmap(i / max(1, len(cats)-1)) for i, cat in enumerate(cats)}
    fig, ax = plt.subplots(figsize=(9, 7), dpi=160)
    for cat in cats:
        sub = df[df[color_col] == cat]
        ax.scatter(sub["UMAP1"], sub["UMAP2"], s=0.25, alpha=0.65, label=str(cat), color=color_map[cat], linewidths=0)
    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_xticks([])
    ax.set_yticks([])
    # Keep legend readable.
    if len(cats) <= 30:
        handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[c], markersize=5, label=str(c)) for c in cats]
        ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=7)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    figure_records.append({"figure_label": title, "path": str(out_png), "exists": Path(out_png).exists(), "size_mb": Path(out_png).stat().st_size/1024/1024 if Path(out_png).exists() else np.nan})
    print("[SAVED FIG]", out_png)
    return out_png

# Temporary plotting columns.
plot_obs = obs.copy()
plot_obs["cluster_for_plot"] = plot_obs["cluster_str"]
plot_obs["copykat_for_plot"] = plot_obs["copykat_pred_clean_for_stats"]
plot_obs["final_group_for_plot"] = plot_obs["final_group_for_stats"]

plot_items = [
    ("final_group_for_plot", "Final malignancy group + CopyKAT/marker consensus"),
    ("copykat_for_plot", "CopyKAT prediction"),
    ("cluster_for_plot", "scVI Leiden cluster"),
]
if annotation_col is not None:
    plot_items.append((annotation_col, "Marker-based annotation"))
if condition_col is not None:
    plot_items.append((condition_col, "Condition"))
if dataset_col is not None:
    plot_items.append((dataset_col, "Dataset"))

for col, title in plot_items:
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", col)
    plot_umap_categorical(
        plot_obs,
        umap,
        col,
        title,
        FIG_DIR / f"UMAP_{safe_name}_{RUN_TAG}.png",
    )

# Barplot final group counts.
fg = obs["final_group_for_stats"].value_counts().reset_index()
fg.columns = ["final_group", "n_cells"]
fig, ax = plt.subplots(figsize=(8, 4), dpi=160)
ax.bar(fg["final_group"].astype(str), fg["n_cells"])
ax.set_ylabel("Cells")
ax.set_title("Final malignancy group counts")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
out = FIG_DIR / f"bar_final_malignancy_group_counts_{RUN_TAG}.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
figure_records.append({"figure_label": "bar_final_malignancy_group_counts", "path": str(out), "exists": out.exists(), "size_mb": out.stat().st_size/1024/1024 if out.exists() else np.nan})

figure_manifest = pd.DataFrame(figure_records)
write_tsv(figure_manifest, METADATA_DIR / f"final_CNV_QC_figure_manifest_{RUN_TAG}.tsv")
display(figure_manifest)


## 4. Statistical validation: CopyKAT enrichment by cluster and final group

These tests use only defined CopyKAT cells (`aneuploid` and `diploid`) and ignore `not_defined/missing` for Fisher tests.

In [ ]:
defined = obs[obs["copykat_pred_clean_for_stats"].isin(["aneuploid", "diploid"])].copy()
print("Defined CopyKAT cells:", defined.shape[0], "of", obs.shape[0])

# Overall association tests.
overall_records = []
for label, col in [
    ("cluster_vs_copykat", "cluster_str"),
    ("final_group_vs_copykat", "final_group_for_stats"),
]:
    tbl = pd.crosstab(defined[col], defined["copykat_pred_clean_for_stats"])
    # Ensure both columns present.
    for c in ["aneuploid", "diploid"]:
        if c not in tbl.columns:
            tbl[c] = 0
    tbl = tbl[["aneuploid", "diploid"]]
    if tbl.shape[0] >= 2 and tbl.values.sum() > 0:
        cv, chi2, p, dof = cramer_v_from_table(tbl.values)
    else:
        cv, chi2, p, dof = np.nan, np.nan, np.nan, np.nan
    overall_records.append({
        "test": label,
        "n_cells_defined": int(tbl.values.sum()),
        "n_categories": tbl.shape[0],
        "chi2": chi2,
        "dof": dof,
        "p_value": p,
        "cramers_v": cv,
    })

overall_tests = pd.DataFrame(overall_records)
overall_tests["q_value_bh"] = bh_fdr(overall_tests["p_value"])
write_tsv(overall_tests, METADATA_DIR / f"final_stat_overall_association_tests_{RUN_TAG}.tsv")
display(overall_tests)

# Per-cluster enrichment: cluster vs all other defined cells.
records = []
all_a = int((defined["copykat_pred_clean_for_stats"] == "aneuploid").sum())
all_d = int((defined["copykat_pred_clean_for_stats"] == "diploid").sum())
for cl, sub in defined.groupby("cluster_str", observed=True):
    a = int((sub["copykat_pred_clean_for_stats"] == "aneuploid").sum())
    d = int((sub["copykat_pred_clean_for_stats"] == "diploid").sum())
    c = all_a - a
    b = all_d - d
    # table: target aneup/dip vs rest aneup/dip
    odds, p = fisher_exact([[a, d], [c, b]], alternative="two-sided")
    records.append({
        "cluster": cl,
        "n_defined": a + d,
        "n_aneuploid": a,
        "n_diploid": d,
        "aneuploid_fraction_defined": a/(a+d) if (a+d)>0 else np.nan,
        "rest_n_aneuploid": c,
        "rest_n_diploid": b,
        "odds_ratio_cluster_vs_rest": safe_odds_ratio(a, d, c, b),
        "fisher_p_value": p,
    })
cluster_enrichment = pd.DataFrame(records)
cluster_enrichment["fisher_q_value_bh"] = bh_fdr(cluster_enrichment["fisher_p_value"])
cluster_enrichment = cluster_enrichment.sort_values("cluster", key=lambda s: pd.to_numeric(s, errors="coerce"))
write_tsv(cluster_enrichment, METADATA_DIR / f"final_stat_cluster_copykat_enrichment_{RUN_TAG}.tsv")
display(cluster_enrichment)

# Final group enrichment.
records = []
for gr, sub in defined.groupby("final_group_for_stats", observed=True):
    a = int((sub["copykat_pred_clean_for_stats"] == "aneuploid").sum())
    d = int((sub["copykat_pred_clean_for_stats"] == "diploid").sum())
    c = all_a - a
    b = all_d - d
    odds, p = fisher_exact([[a, d], [c, b]], alternative="two-sided")
    records.append({
        "final_group": gr,
        "n_defined": a + d,
        "n_aneuploid": a,
        "n_diploid": d,
        "aneuploid_fraction_defined": a/(a+d) if (a+d)>0 else np.nan,
        "odds_ratio_group_vs_rest": safe_odds_ratio(a, d, c, b),
        "fisher_p_value": p,
    })
group_enrichment = pd.DataFrame(records)
group_enrichment["fisher_q_value_bh"] = bh_fdr(group_enrichment["fisher_p_value"])
group_enrichment = group_enrichment.sort_values("aneuploid_fraction_defined", ascending=False)
write_tsv(group_enrichment, METADATA_DIR / f"final_stat_group_copykat_enrichment_{RUN_TAG}.tsv")
display(group_enrichment)


## 5. Statistical validation: tumor-like and cycling clusters versus core reference clusters

In [ ]:
# Tumor/cycling clusters versus pooled core reference clusters.
ref_mask = defined["cluster_str"].isin(CORE_REFERENCE_CLUSTERS)
ref = defined[ref_mask]
if ref.empty:
    print("[WARNING] No core reference clusters found; skipping tumor vs reference Fisher tests.")
    tumor_vs_ref = pd.DataFrame()
else:
    ref_a = int((ref["copykat_pred_clean_for_stats"] == "aneuploid").sum())
    ref_d = int((ref["copykat_pred_clean_for_stats"] == "diploid").sum())
    records = []
    target_clusters = sorted(MALIGNANT_CLUSTERS | CYCLING_CANDIDATE_CLUSTERS, key=lambda x: int(x) if x.isdigit() else 999)
    for cl in target_clusters:
        sub = defined[defined["cluster_str"] == cl]
        if sub.empty:
            continue
        a = int((sub["copykat_pred_clean_for_stats"] == "aneuploid").sum())
        d = int((sub["copykat_pred_clean_for_stats"] == "diploid").sum())
        odds, p = fisher_exact([[a, d], [ref_a, ref_d]], alternative="two-sided")
        records.append({
            "target_cluster": cl,
            "target_role": "cycling_candidate" if cl in CYCLING_CANDIDATE_CLUSTERS else "malignant_like",
            "target_n_defined": a+d,
            "target_n_aneuploid": a,
            "target_n_diploid": d,
            "target_aneuploid_fraction_defined": a/(a+d) if (a+d)>0 else np.nan,
            "reference_clusters": ",".join(sorted(CORE_REFERENCE_CLUSTERS)),
            "reference_n_defined": ref_a+ref_d,
            "reference_n_aneuploid": ref_a,
            "reference_n_diploid": ref_d,
            "reference_aneuploid_fraction_defined": ref_a/(ref_a+ref_d) if (ref_a+ref_d)>0 else np.nan,
            "odds_ratio_target_vs_reference": safe_odds_ratio(a, d, ref_a, ref_d),
            "fisher_p_value": p,
        })
    tumor_vs_ref = pd.DataFrame(records)
    tumor_vs_ref["fisher_q_value_bh"] = bh_fdr(tumor_vs_ref["fisher_p_value"])

write_tsv(tumor_vs_ref, METADATA_DIR / f"final_stat_tumor_vs_reference_copykat_fisher_{RUN_TAG}.tsv")
display(tumor_vs_ref)


## 6. Sample-level summaries and condition-level tests

This section avoids using only per-cell p-values by aggregating cell states at sample level first.

In [ ]:
# Prepare sample-level data.
if sample_col is None:
    obs["sample_for_stats"] = "all_cells_single_sample_missing_sample_id"
    sample_col_use = "sample_for_stats"
else:
    sample_col_use = sample_col

condition_col_use = condition_col if condition_col is not None else None
dataset_col_use = dataset_col if dataset_col is not None else None
patient_col_use = patient_col if patient_col is not None else None

groupby_cols = [sample_col_use]
for c in [patient_col_use, dataset_col_use, condition_col_use]:
    if c is not None and c not in groupby_cols:
        groupby_cols.append(c)

sample_counts = (
    obs.groupby(groupby_cols + ["final_group_for_stats"], observed=True)
       .size().reset_index(name="n_cells")
)
sample_totals = obs.groupby(groupby_cols, observed=True).size().reset_index(name="sample_n_cells")
sample_props = sample_counts.merge(sample_totals, on=groupby_cols, how="left")
sample_props["fraction_of_sample"] = sample_props["n_cells"] / sample_props["sample_n_cells"]
write_tsv(sample_props, METADATA_DIR / f"final_CNV_QC_sample_group_proportions_{RUN_TAG}.tsv")
display(sample_props.head(20))

# Condition-level tests on sample-level proportions.
condition_tests = []
if condition_col_use is not None:
    for gr, sub in sample_props.groupby("final_group_for_stats", observed=True):
        cond_groups = []
        cond_names = []
        for cond, ss in sub.groupby(condition_col_use, observed=True):
            vals = ss["fraction_of_sample"].dropna().astype(float).values
            if len(vals) > 0:
                cond_groups.append(vals)
                cond_names.append(str(cond))
        if len(cond_groups) >= 2:
            # Overall Kruskal-Wallis if at least two condition groups have data.
            try:
                stat, p = kruskal(*cond_groups)
            except Exception:
                stat, p = np.nan, np.nan
            condition_tests.append({
                "test_type": "kruskal_overall_by_condition",
                "final_group": gr,
                "condition_a": "ALL",
                "condition_b": "ALL",
                "n_condition_groups": len(cond_groups),
                "n_samples_total": int(sum(len(x) for x in cond_groups)),
                "statistic": stat,
                "p_value": p,
            })
            # Pairwise Mann-Whitney.
            for i in range(len(cond_groups)):
                for j in range(i+1, len(cond_groups)):
                    va, vb = cond_groups[i], cond_groups[j]
                    if len(va) >= 2 and len(vb) >= 2:
                        try:
                            stat, p = mannwhitneyu(va, vb, alternative="two-sided")
                        except Exception:
                            stat, p = np.nan, np.nan
                        condition_tests.append({
                            "test_type": "mannwhitney_pairwise_by_condition",
                            "final_group": gr,
                            "condition_a": cond_names[i],
                            "condition_b": cond_names[j],
                            "n_condition_groups": 2,
                            "n_samples_total": int(len(va) + len(vb)),
                            "statistic": stat,
                            "p_value": p,
                        })
else:
    print("[SKIP] No condition column found; sample-level condition tests skipped.")

condition_tests = pd.DataFrame(condition_tests)
if not condition_tests.empty:
    condition_tests["q_value_bh"] = bh_fdr(condition_tests["p_value"])
write_tsv(condition_tests, METADATA_DIR / f"final_stat_condition_sample_level_tests_{RUN_TAG}.tsv")
display(condition_tests.head(50))


## 7. Output manifest

In [ ]:
output_paths = [
    METADATA_DIR / f"final_CNV_QC_readiness_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_obs_column_report_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_group_counts_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_cluster_group_copykat_summary_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_sample_group_proportions_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_overall_association_tests_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_cluster_copykat_enrichment_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_group_copykat_enrichment_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_tumor_vs_reference_copykat_fisher_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_condition_sample_level_tests_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_figure_manifest_{RUN_TAG}.tsv",
]
manifest = pd.DataFrame([
    {"output_label": p.stem, "path": str(p), "exists": p.exists(), "size_mb": p.stat().st_size/1024/1024 if p.exists() else np.nan}
    for p in output_paths
])
write_tsv(manifest, METADATA_DIR / f"final_CNV_QC_statistical_output_manifest_{RUN_TAG}.tsv")
display(manifest)
print("DONE")
